# مسئلهٔ ۱ — V2 / مرحلهٔ ۴: استخراج و cache قطعی فریم‌های sequence

این نوت‌بوک فقط timestampهای ثبت‌شده در `sequence_manifest_v2.csv` را decode می‌کند. برای هر sequence، ۱۶ فریم در مسیر جداگانهٔ `processed_v2` ذخیره می‌شوند. هیچ MP4، split یا خروجی V1 تغییر یا حذف نمی‌شود.

پیش‌پردازش قطعی V2 اولیه: BGR → RGB، بدون crop، حفظ نسبت تصویر با letterbox و خروجی `224×320`. Augmentation تصادفی در این مرحله اعمال نمی‌شود؛ آن فقط هنگام آموزش train خواهد بود.

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_ROOT = Path(r'P:\NexarCollisionData')
SEQUENCE_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2.csv'
CACHE_ROOT = DATA_ROOT / 'processed_v2' / 'frames16_wide_224x320_v2_w2'
FRAME_INDEX_PATH = DATA_ROOT / 'frame_cache_index_v2.csv'
SEQUENCE_STATUS_PATH = DATA_ROOT / 'sequence_cache_status_v2.csv'
SUMMARY_PATH = DATA_ROOT / 'frame_cache_summary_v2.json'
PREVIEW_PATH = DATA_ROOT / 'frame_preview_v2.jpg'

TARGET_HEIGHT = 224
TARGET_WIDTH = 320
NUM_FRAMES = 16
JPEG_QUALITY = 95
PREPROCESSING_VERSION = 'v2_w2_rgb_letterbox_replicate_224x320'

assert SEQUENCE_MANIFEST_PATH.exists(), f'Run notebook 08 first: {SEQUENCE_MANIFEST_PATH}'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Frame cache root: {CACHE_ROOT}')

Frame cache root: P:\NexarCollisionData\processed_v2\frames16_wide_224x320_v2_w2


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sequence_manifest = pd.read_csv(SEQUENCE_MANIFEST_PATH)
timestamp_columns = [f'timestamp_{index:02d}' for index in range(NUM_FRAMES)]
required_columns = {
    'sequence_id', 'video_id', 'video_path', 'label', 'split', 'duration',
    'window_start', 'window_end', 'num_frames', *timestamp_columns,
}
assert required_columns.issubset(sequence_manifest.columns), sorted(required_columns - set(sequence_manifest.columns))

sequence_manifest = sequence_manifest.copy()
sequence_manifest['video_id'] = sequence_manifest['video_id'].astype(str)
sequence_manifest['label'] = sequence_manifest['label'].astype(int)
sequence_manifest['num_frames'] = sequence_manifest['num_frames'].astype(int)
assert len(sequence_manifest) == 600
assert sequence_manifest['sequence_id'].is_unique
assert sequence_manifest['video_id'].is_unique
assert sequence_manifest['num_frames'].eq(NUM_FRAMES).all()
assert sequence_manifest['split'].isin(['train', 'validation']).all()

print('Sequences to cache:')
display(pd.crosstab(sequence_manifest['split'], sequence_manifest['label']))

Sequences to cache:


label,0,1
split,,
train,240,240
validation,60,60


In [3]:
def class_name(label: int) -> str:
    return 'positive' if int(label) == 1 else 'negative'

def output_dir_for_sequence(row: pd.Series) -> Path:
    return CACHE_ROOT / row.split / class_name(row.label) / f'{int(row.video_id):05d}'

def output_path_for_frame(row: pd.Series, frame_index: int) -> Path:
    return output_dir_for_sequence(row) / f'frame_{frame_index:02d}.jpg'

def resize_letterbox_rgb(frame_rgb: np.ndarray) -> np.ndarray:
    """Preserve the dashcam aspect ratio; padding uses edge replication, not black bars."""
    height, width = frame_rgb.shape[:2]
    scale = min(TARGET_WIDTH / width, TARGET_HEIGHT / height)
    resized_width = max(1, int(round(width * scale)))
    resized_height = max(1, int(round(height * scale)))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    resized = cv2.resize(frame_rgb, (resized_width, resized_height), interpolation=interpolation)

    pad_x = TARGET_WIDTH - resized_width
    pad_y = TARGET_HEIGHT - resized_height
    left, right = pad_x // 2, pad_x - (pad_x // 2)
    top, bottom = pad_y // 2, pad_y - (pad_y // 2)
    return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_REPLICATE)

def decode_rgb_at_timestamp(cap: cv2.VideoCapture, timestamp: float, fps: float) -> tuple[np.ndarray | None, str, float]:
    """OpenCV fallback seeks are recorded so a failed read is never hidden."""
    step = 1.0 / fps if np.isfinite(fps) and fps > 0 else 1.0 / 30.0
    for attempt, offset in enumerate((0.0, step, -step, 2 * step)):
        target_timestamp = max(0.0, timestamp + offset)
        cap.set(cv2.CAP_PROP_POS_MSEC, target_timestamp * 1000.0)
        ok, frame_bgr = cap.read()
        if ok and frame_bgr is not None:
            status = 'exact' if attempt == 0 else f'seek_fallback_{attempt}'
            return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB), status, target_timestamp
    return None, 'decode_failed', np.nan

def complete_existing_cache(row: pd.Series) -> bool:
    """Skip only an already complete and readable 16-frame cache."""
    for frame_index in range(NUM_FRAMES):
        output_path = output_path_for_frame(row, frame_index)
        if not output_path.is_file():
            return False
        cached_bgr = cv2.imread(str(output_path), cv2.IMREAD_COLOR)
        if cached_bgr is None or cached_bgr.shape[:2] != (TARGET_HEIGHT, TARGET_WIDTH):
            return False
    return True

## اجرای extraction

در اجرای نخست، ۹۶۰۰ فریم decode و ذخیره می‌شوند و بسته به دیسک/CPU چندین دقیقه زمان می‌گیرد. اجرای دوباره ویدئو یا فریم تکراری تولید نمی‌کند: فقط sequenceهایی که ۱۶ JPEG سالم دارند skip می‌شوند.

In [4]:
frame_records = []
sequence_status_records = []

for _, row in tqdm(sequence_manifest.iterrows(), total=len(sequence_manifest), desc='Caching sequence frames'):
    output_dir = output_dir_for_sequence(row)
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamps = [float(row[column]) for column in timestamp_columns]

    if complete_existing_cache(row):
        for frame_index, timestamp in enumerate(timestamps):
            frame_records.append({
                'sequence_id': row.sequence_id, 'video_id': row.video_id, 'video_path': row.video_path,
                'label': row.label, 'split': row.split, 'frame_index': frame_index,
                'requested_timestamp': timestamp, 'resolved_timestamp': timestamp,
                'frame_path': str(output_path_for_frame(row, frame_index)),
                'frame_valid': True, 'decode_status': 'existing_cache',
                'width': TARGET_WIDTH, 'height': TARGET_HEIGHT,
                'preprocessing_version': PREPROCESSING_VERSION,
            })
        sequence_status_records.append({
            'sequence_id': row.sequence_id, 'video_id': row.video_id, 'label': row.label, 'split': row.split,
            'cache_status': 'skipped_complete_cache', 'valid_frames': NUM_FRAMES,
            'invalid_frames': 0, 'error_reason': None,
        })
        continue

    cap = cv2.VideoCapture(str(row.video_path))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) if cap.isOpened() else np.nan
    previous_rgb = None
    valid_frames = 0
    invalid_frames = 0
    errors = []

    if not cap.isOpened():
        errors.append('cannot_open')

    for frame_index, timestamp in enumerate(timestamps):
        output_path = output_path_for_frame(row, frame_index)
        frame_rgb, decode_status, resolved_timestamp = (None, 'cannot_open', np.nan)
        if cap.isOpened():
            frame_rgb, decode_status, resolved_timestamp = decode_rgb_at_timestamp(cap, timestamp, fps)

        frame_valid = frame_rgb is not None
        if frame_rgb is None and previous_rgb is not None:
            frame_rgb = previous_rgb.copy()
            decode_status = 'repeated_previous_after_decode_failure'
        elif frame_rgb is None:
            errors.append(f'frame_{frame_index:02d}:{decode_status}')

        if frame_rgb is not None:
            processed_rgb = resize_letterbox_rgb(frame_rgb)
            write_ok = cv2.imwrite(
                str(output_path), cv2.cvtColor(processed_rgb, cv2.COLOR_RGB2BGR),
                [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY],
            )
            if not write_ok:
                errors.append(f'frame_{frame_index:02d}:write_failed')
                frame_valid = False
                frame_path = None
            else:
                frame_path = str(output_path)
                previous_rgb = frame_rgb
        else:
            frame_path = None

        valid_frames += int(frame_valid)
        invalid_frames += int(not frame_valid)
        frame_records.append({
            'sequence_id': row.sequence_id, 'video_id': row.video_id, 'video_path': row.video_path,
            'label': row.label, 'split': row.split, 'frame_index': frame_index,
            'requested_timestamp': timestamp, 'resolved_timestamp': resolved_timestamp,
            'frame_path': frame_path, 'frame_valid': frame_valid, 'decode_status': decode_status,
            'width': TARGET_WIDTH, 'height': TARGET_HEIGHT,
            'preprocessing_version': PREPROCESSING_VERSION,
        })

    cap.release()
    sequence_status_records.append({
        'sequence_id': row.sequence_id, 'video_id': row.video_id, 'label': row.label, 'split': row.split,
        'cache_status': 'processed', 'valid_frames': valid_frames, 'invalid_frames': invalid_frames,
        'error_reason': None if not errors else ';'.join(errors),
    })

frame_cache_index = pd.DataFrame(frame_records)
sequence_cache_status = pd.DataFrame(sequence_status_records)
frame_cache_index.to_csv(FRAME_INDEX_PATH, index=False)
sequence_cache_status.to_csv(SEQUENCE_STATUS_PATH, index=False)

print(f'Frame index saved: {FRAME_INDEX_PATH}')
print(f'Sequence cache status saved: {SEQUENCE_STATUS_PATH}')

Caching sequence frames: 100%|██████████| 600/600 [50:16<00:00,  5.03s/it]

Frame index saved: P:\NexarCollisionData\frame_cache_index_v2.csv
Sequence cache status saved: P:\NexarCollisionData\sequence_cache_status_v2.csv


In [5]:
assert len(frame_cache_index) == len(sequence_manifest) * NUM_FRAMES
assert len(sequence_cache_status) == len(sequence_manifest)
assert frame_cache_index.groupby('sequence_id').size().eq(NUM_FRAMES).all()

valid_frame_count = int(frame_cache_index['frame_valid'].sum())
invalid_frame_count = int((~frame_cache_index['frame_valid']).sum())
invalid_sequence_count = int((sequence_cache_status['invalid_frames'] > 0).sum())
shape_errors = 0
for frame_path in frame_cache_index.loc[frame_cache_index['frame_valid'], 'frame_path']:
    image_bgr = cv2.imread(str(frame_path), cv2.IMREAD_COLOR)
    if image_bgr is None or image_bgr.shape[:2] != (TARGET_HEIGHT, TARGET_WIDTH):
        shape_errors += 1

summary = {
    'source_sequence_manifest': str(SEQUENCE_MANIFEST_PATH),
    'cache_root': str(CACHE_ROOT),
    'sequences': int(len(sequence_manifest)),
    'frames_expected': int(len(sequence_manifest) * NUM_FRAMES),
    'frames_valid': valid_frame_count,
    'frames_invalid': invalid_frame_count,
    'sequences_with_invalid_frames': invalid_sequence_count,
    'image_shape_errors': int(shape_errors),
    'target_height': TARGET_HEIGHT,
    'target_width': TARGET_WIDTH,
    'resize_mode': 'letterbox_with_replicated_edge_padding',
    'color_order_before_model_normalization': 'RGB',
    'preprocessing_version': PREPROCESSING_VERSION,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Summary saved: {SUMMARY_PATH}')
display(pd.DataFrame([summary]))
display(sequence_cache_status.groupby(['split', 'label', 'cache_status'])[['valid_frames', 'invalid_frames']].sum())

# Do not train until every requested frame is present and readable.
assert invalid_frame_count == 0, 'Inspect frame_cache_index_v2.csv before training.'
assert shape_errors == 0, 'Some cached images have an unexpected shape.'

Summary saved: P:\NexarCollisionData\frame_cache_summary_v2.json


,source_sequence_manifest,cache_root,sequences,frames_expected,frames_valid,frames_invalid,sequences_with_invalid_frames,image_shape_errors,target_height,target_width,resize_mode,color_order_before_model_normalization,preprocessing_version
0,P:\NexarCollisionData\sequence_manifest_v2.csv,P:\NexarCollisionData\processed_v2\frames16_wi...,600,9600,9600,0,0,0,224,320,letterbox_with_replicated_edge_padding,RGB,v2_w2_rgb_letterbox_replicate_224x320


valid_frames  invalid_frames
split      label cache_status                              
train      0     processed             3840               0
           1     processed             3840               0
validation 0     processed              960               0
           1     processed              960               0

In [6]:
def make_preview(label: int, title: str) -> np.ndarray:
    candidate = frame_cache_index.loc[
        frame_cache_index['label'].eq(label) & frame_cache_index['frame_valid']
    ].sort_values(['sequence_id', 'frame_index']).groupby('sequence_id').head(4)
    images = []
    for _, frame_row in candidate.head(4).iterrows():
        image_bgr = cv2.imread(str(frame_row.frame_path), cv2.IMREAD_COLOR)
        cv2.putText(image_bgr, f'{title}  t={frame_row.requested_timestamp:.2f}s', (8, 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
        images.append(image_bgr)
    return cv2.hconcat(images)

preview_bgr = cv2.vconcat([make_preview(1, 'positive'), make_preview(0, 'negative')])
cv2.imwrite(str(PREVIEW_PATH), preview_bgr, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
print(f'Preview saved: {PREVIEW_PATH}')
display(PREVIEW_PATH)

Preview saved: P:\NexarCollisionData\frame_preview_v2.jpg


WindowsPath('P:/NexarCollisionData/frame_preview_v2.jpg')

## شرط پایان مرحلهٔ ۴

انتظار داریم `frames_expected = frames_valid = 9600`، هیچ frame mask نامعتبر و هیچ خطای shape نداشته باشیم. سپس Dataset در سطح sequence ساخته می‌شود: هر نمونه `[16, 3, 224, 320]` و فقط یک label ویدئویی دارد.